In [ ]:
!pip uninstall -y langchain langchain-community langchain-core
!pip install langchain==0.1.17 langchain-community faiss-cpu transformers sentence-transformers pypdf

Found existing installation: langchain 0.1.17
Uninstalling langchain-0.1.17:
  Successfully uninstalled langchain-0.1.17
Found existing installation: langchain-community 0.0.38
Uninstalling langchain-community-0.0.38:
  Successfully uninstalled langchain-community-0.0.38
Found existing installation: langchain-core 0.1.53
Uninstalling langchain-core-0.1.53:
  Successfully uninstalled langchain-core-0.1.53
  Using cached langchain-0.1.17-py3-none-any.whl.metadata (13 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.0.38-py3-none-any.whl.metadata (8.7 kB)
  Using cached langchain_core-0.1.53-py3-none-any.whl.metadata (5.9 kB)
Using cached langchain-0.1.17-py3-none-any.whl (867 kB)
Using cached langchain_community-0.0.38-py3-none-any.whl (2.0 MB)
Using cached langchain_core-0.1.53-py3-none-any.whl (303 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behavi

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Artificial_intelligence_in_India.pdf to Artificial_intelligence_in_India.pdf
Saving Climate_change_in_India.pdf to Climate_change_in_India.pdf
Saving COVID-19_pandemic.pdf to COVID-19_pandemic.pdf
Saving ISRO.pdf to ISRO.pdf
Saving Temples_of_modern_India.pdf to Temples_of_modern_India.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import os

documents = []

# Create 'data' directory if it doesn't exist
os.makedirs("data", exist_ok=True)

# Move uploaded files from root to 'data' directory
# Assuming 'uploaded' variable contains the names of the uploaded files in the current context
# (this would typically come from a previous cell like files.upload())
# Since 'uploaded' is a dictionary from files.upload(), its keys are the filenames.
for filename in uploaded.keys():
    if filename.endswith(".pdf"):
        # Check if the file is already in 'data' to prevent error on re-run
        if not os.path.exists(os.path.join("data", filename)):
            os.rename(filename, os.path.join("data", filename))
            print(f"✅ Moved: {filename} to data/{filename}")

for file in os.listdir("data"):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(f"data/{file}")
        documents.extend(loader.load())

print("✅ Documents loaded:", len(documents))

✅ Documents loaded: 232


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("✅ Total chunks:", len(chunks))

✅ Total chunks: 974


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("✅ Embeddings ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embeddings ready


In [ ]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(chunks, embeddings)

db.save_local("faiss_index")

print("✅ Database created")

✅ Database created


In [ ]:
# Cell 7 — Load FAISS and set up retriever
# k=4 means retrieve top 4 most relevant chunks (configurable)

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

TOP_K = 4  # Change this number to get more or fewer results

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
retriever = db.as_retriever(search_kwargs={"k": TOP_K})

print(f"✅ Retriever ready! Will fetch top {TOP_K} chunks per question")

In [ ]:
# Cell 8B — Generate proper answer using HuggingFace LLM (FREE - no API key needed)
from transformers import pipeline
import textwrap

# Load a better free model for question answering
qa_pipeline = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2"
)

def ask(question):
    print(f"\n❓ Question: {question}")
    print("🔍 Searching documents...\n")

    # Get relevant chunks
    docs = retriever.get_relevant_documents(question)

    # Combine top chunks as context
    context = " ".join([doc.page_content for doc in docs[:3]])
    context = context[:2000]  # limit context size

    # Generate answer using QA model
    try:
        result = qa_pipeline(question=question, context=context)
        answer = result['answer']
        score = result['score']

        print(f"🤖 Answer: {answer}")
        print(f"   Confidence: {score:.0%}")
    except:
        answer = "Could not find a specific answer in the documents."
        print(f"🤖 Answer: {answer}")

    # Show sources
    print(f"\n📚 Sources:")
    seen = set()
    for doc in docs:
        src = doc.metadata.get('source', 'Unknown')
        page = doc.metadata.get('page', 'N/A')
        key = f"{src} | Page {page}"
        if key not in seen:
            print(f"   → {key}")
            seen.add(key)
        # Show chunk preview
        preview = doc.page_content[:200].replace('\n', ' ')
        print(f"     Preview: {preview}...")
    print("\n" + "="*60)

print("✅ Q&A Bot with LLM ready!")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

✅ Q&A Bot with LLM ready!


In [9]:
# Cell 8 — Ask questions directly without GPT-2
# This retrieves relevant chunks and shows them with sources clearly

def ask(question):
    print(f"\n Question: {question}")
    print(" Searching your documents...\n")

    # Get relevant chunks
    docs = retriever.get_relevant_documents(question)

    print(" Most relevant information found:\n")
    seen = set()
    for i, doc in enumerate(docs):
        src = doc.metadata.get('source', 'Unknown')
        page = doc.metadata.get('page', 'N/A')
        key = f"{src}_p{page}"
        # Show content preview
        preview = doc.page_content[:300].replace('\n', ' ')
        print(f"[{i+1}] From: {src} | Page {page}")
        print(f"     {preview}...")
        print()
        seen.add(key)

    print("Sources used:")
    for key in seen:
        print(f"   → {key.replace('_p', ' | Page ')}")
    print("\n" + "="*60)

print("✅ Bot ready! Running questions...\n")
print("="*60)

# Ask all 5 questions
ask("what is machine learning?")
ask("What causes climate change?")
ask("What are the symptoms of COVID-19?")
ask("What is ISRO and what has it achieved?")
ask("What are famous temples in India?")

✅ Bot ready! Running questions...


 Question: what is machine learning?
 Searching your documents...

 Most relevant information found:

[1] From: data/Artificial_intelligence_in_India.pdf | Page 24
     Theme Institute Project Machine unlearning IIT Jodhpur Develop method for targeted unlearning in open-source generative foundation models while minimizing negative impact on overall model performance. Synthetic data generation IIT Roorkee Design and develop method for generating synthetic data for m...

[2] From: data/Artificial_intelligence_in_India.pdf | Page 26
     16. Chandru, Vijay (21 December 2020). "The academic legacy of AI in India" (https://www.histo riansofthenow.org/post/the-academic-legacy-of-ai-in-india). Historians of the Now. Retrieved 29 January 2025. 17. "Landscape of Artificial Intelligence / Machine Learning Research in India" (https://itihaa...

[3] From: data/Artificial_intelligence_in_India.pdf | Page 12
     India.[133][134][135] India's Ministry of Electroni

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


In [10]:
# This question is NOT in our documents — bot should find nothing relevant
ask("What is the current price of gold in India today?")


 Question: What is the current price of gold in India today?
 Searching your documents...

 Most relevant information found:

[1] From: data/Artificial_intelligence_in_India.pdf | Page 34
     119. "With robust and high end Common computing facility in place, India all set to launch its own safe & secure indigenous AI model at affordable cost soon: Shri Ashwini Vaishnaw" (https:// pib.gov.in/PressReleasePage.aspx?PRID=2097709). Press Information Bureau. Ministry of Electronics & IT, Gover...

[2] From: data/ISRO.pdf | Page 44
     155. "Shubhanshu Shukla: The Isro pilot taking India back to space after 41 years" (https://www.b bc.com/news/articles/ce80glkl7nno). BBC. 3 June 2025. 156. "Indian astronaut Shubhanshu Shukla's space mission has a whopping price tag" (https://w ww.indiatoday.in/science/story/indian-astronaut-shubha...

[3] From: data/Artificial_intelligence_in_India.pdf | Page 30
     71-751X). Retrieved 8 February 2025. 67. "PM Modi calls for global regulation of AI" (http

In [11]:
# Live Q&A — type your question and press Enter
while True:
    question = input(" Ask a question (type 'quit' to stop): ").strip()

    if question.lower() == 'quit':
        print(" Goodbye!")
        break

    if not question:
        continue

    ask(question)

 Ask a question (type 'quit' to stop): what is artificial intelligence?

 Question: what is artificial intelligence?
 Searching your documents...

 Most relevant information found:

[1] From: data/Artificial_intelligence_in_India.pdf | Page 3
     NITI Aayog Problem to Solution Incubation Test Bed will be established. Microsoft will also help develop AI-assisted models for diabetic retinopathy screening.[49][50] NITI Aayog drafted a proposal in 2019 to establish an institutional framework for AI. A cabinet note has been issued proposing to al...

[2] From: data/Artificial_intelligence_in_India.pdf | Page 12
     India.[133][134][135] India's Ministry of Electronics and Information Technology and Japan's Ministry of Economy, Trade, and Industry signed a memorandum of cooperation on 29 October 2018, which outlines the two nations' cooperation in AI and IoT as part of India-Japan Cooperation on Digital Partner...

[3] From: data/Artificial_intelligence_in_India.pdf | Page 7
     Pune IIT 